In [1]:
import numpy as np
import pandas as pd
import random
import os
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import faiss
import torch
from sentence_transformers import SentenceTransformer

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используется устройство: {device}")

Используется устройство: cpu


In [3]:
kb_folder = "data/knowledge_base/"
file_paths = sorted(glob.glob(os.path.join(kb_folder, "*.txt")))
print(f"Найдено файлов: {len(file_paths)}")

knowledge_base = []
for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    # Первая строка — заголовок, остальное — текст
    lines = content.split('\n', 1)
    title = lines[0].strip()
    text = lines[1].strip() if len(lines) > 1 else ""
    doc_id = int(os.path.basename(file_path).split('_')[1].split('.')[0])
    knowledge_base.append({"id": doc_id, "title": title, "text": text})

print(f"Загружено документов: {len(knowledge_base)}")
print("\nПримеры документов (заголовки):")
for i, doc in enumerate(knowledge_base[:5]):
    print(f"{i+1}. {doc['title']}")

Найдено файлов: 20
Загружено документов: 20

Примеры документов (заголовки):
1. Эмбеддинги текстов
2. FAISS и поиск ближайших соседей
3. Чанкинг и overlap при разбиении текста
4. Косинусное сходство и его применение
5. Retrieval-Augmented Generation (RAG)


In [4]:
def chunk_text(text, chunk_size_words=50, overlap_words=10):
    words = text.split()
    chunks = []
    step = chunk_size_words - overlap_words
    if step <= 0:
        step = chunk_size_words
    for start in range(0, len(words), step):
        end = start + chunk_size_words
        chunk_words = words[start:end]
        if len(chunk_words) < 5:
            continue
        chunks.append(' '.join(chunk_words))
    return chunks

CHUNK_SIZE = 50
OVERLAP = 10

chunks = []
for doc in knowledge_base:
    doc_chunks = chunk_text(doc['text'], CHUNK_SIZE, OVERLAP)
    for idx, chunk_text_str in enumerate(doc_chunks):
        chunks.append({
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_index': idx,
            'text': chunk_text_str
        })

print(f"Получено чанков: {len(chunks)}")
print("\nПример чанка:")
print(chunks[0]['text'][:150], "...")

Получено чанков: 99

Пример чанка:
Эмбеддинги текстов — это плотные векторные представления, которые сохраняют семантический смысл слов, предложений или целых документов. Они получаются ...


In [5]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
model.to(device)

chunk_texts = [c['text'] for c in chunks]
embeddings = model.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=True)
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"Размерность эмбеддингов: {dim}")
print(f"Количество векторов в индексе: {index.ntotal}")

def search(query, idx, chks, top_k=3):
    """Поиск с использованием переданных индекса и списка чанков"""
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = idx.search(q_emb, top_k)
    results = []
    for score, idx_i in zip(scores[0], indices[0]):
        results.append({'chunk': chks[idx_i], 'score': float(score)})
    return results

test_queries = ["Что такое эмбеддинги?", "Как работает FAISS?", "Что такое overlap?"]
for q in test_queries:
    print(f"\nЗапрос: {q}")
    for res in search(q, index, chunks, top_k=2):
        print(f"  - {res['chunk']['doc_title']} (score={res['score']:.3f})")

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

C:\Users\varva\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\varva\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Размерность эмбеддингов: 384
Количество векторов в индексе: 99

Запрос: Что такое эмбеддинги?
  - Эмбеддинги текстов (score=0.440)
  - Эмбеддинги текстов (score=0.394)

Запрос: Как работает FAISS?
  - Индексирование в FAISS: типы индексов (score=0.447)
  - Чанкинг и overlap при разбиении текста (score=0.412)

Запрос: Что такое overlap?
  - Параметры чанкинга и их влияние на метрики (score=0.613)
  - Чанкинг и overlap при разбиении текста (score=0.552)


In [12]:
queries_eval = [
    {"query": "Как получить векторное представление текста?", "expected_title": "Эмбеддинги текстов"},
    {"query": "Библиотека для быстрого поиска ближайших соседей", "expected_title": "FAISS и поиск ближайших соседей"},
    {"query": "Зачем разбивать документы на фрагменты с перекрытием?", "expected_title": "Чанкинг и overlap при разбиении текста"},
    {"query": "Мера сходства, основанная на угле между векторами", "expected_title": "Косинусное сходство и его применение"},
    {"query": "Архитектура, объединяющая поиск и генерацию", "expected_title": "Retrieval-Augmented Generation (RAG)"},
    {"query": "Как оценить, насколько хорошо поиск находит нужный документ?", "expected_title": "Метрики оценки качества retrieval: hit, recall, MRR"},
    {"query": "Модель для русскоязычных эмбеддингов предложений", "expected_title": "Sentence‑transformers модели для русского языка"},
    {"query": "Классический метод векторизации, основанный на частоте терминов", "expected_title": "TF‑IDF как альтернатива эмбеддингам"},
    {"query": "Что происходит при добавлении новых документов в индекс?", "expected_title": "Обновление базы знаний и переиндексация"},
    {"query": "Упрощённая версия RAG для демонстрации механики", "expected_title": "Mini‑RAG пайплайн"}
]

def evaluate_retrieval(queries, idx, chks, model, k=3):
    results = []
    for q in queries:
        query = q["query"]
        expected = q["expected_title"]
        q_emb = model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q_emb)
        scores, indices = idx.search(q_emb, k)
        retrieved_titles = [chks[i]['doc_title'] for i in indices[0]]
        hit = 1 if expected in retrieved_titles else 0
        recall = hit
        mrr = 0.0
        rank = -1
        for r, title in enumerate(retrieved_titles, start=1):
            if title == expected:
                mrr = 1.0 / r
                rank = r
                break
        results.append({
            'query': query,
            'expected_source': expected,
            'retrieved_sources': ', '.join(retrieved_titles),
            'hit_at_k': hit,
            'recall_at_k': recall,
            'mrr_at_k': mrr,
            'rank_of_first_relevant': rank
        })
    return pd.DataFrame(results)

df_eval = evaluate_retrieval(queries_eval, index, chunks, model, k=3)
print("Оценка retrieval (k=3):")
print(df_eval[['query', 'expected_source', 'hit_at_k', 'mrr_at_k']].to_string(index=False))
print(f"\nСредний hit@3: {df_eval['hit_at_k'].mean():.3f}")
print(f"Средний recall@3: {df_eval['recall_at_k'].mean():.3f}")
print(f"Средний MRR@3: {df_eval['mrr_at_k'].mean():.3f}")

os.makedirs('artifacts', exist_ok=True)
df_eval[['query', 'expected_source', 'retrieved_sources', 'hit_at_k', 'rank_of_first_relevant']].to_csv(
    'artifacts/retrieval_eval.csv', 
    index=False, 
    encoding='utf-8-sig'
)

Оценка retrieval (k=3):
                                                          query                                     expected_source  hit_at_k  mrr_at_k
                   Как получить векторное представление текста?                                  Эмбеддинги текстов         1       1.0
               Библиотека для быстрого поиска ближайших соседей                     FAISS и поиск ближайших соседей         0       0.0
          Зачем разбивать документы на фрагменты с перекрытием?              Чанкинг и overlap при разбиении текста         1       0.5
              Мера сходства, основанная на угле между векторами                Косинусное сходство и его применение         1       1.0
                    Архитектура, объединяющая поиск и генерацию                Retrieval-Augmented Generation (RAG)         0       0.0
   Как оценить, насколько хорошо поиск находит нужный документ? Метрики оценки качества retrieval: hit, recall, MRR         0       0.0
               Модель дл

In [7]:
df_hit1 = evaluate_retrieval(queries_eval, index, chunks, model, k=1)
hit1 = df_hit1['hit_at_k'].mean()
print(f"Средний hit@1 = {hit1:.3f}")
print(f"Средний hit@3 = {df_eval['hit_at_k'].mean():.3f}")
print("Вывод: увеличение top_k с 1 до 3 повышает вероятность нахождения релевантного документа,")
print("что критически важно для RAG, так как генератор может использовать информацию из любого из top-k чанков.")

Средний hit@1 = 0.400
Средний hit@3 = 0.600
Вывод: увеличение top_k с 1 до 3 повышает вероятность нахождения релевантного документа,
что критически важно для RAG, так как генератор может использовать информацию из любого из top-k чанков.


In [13]:
new_docs = [
    {"id": 21, "title": "Ранжирование в поиске и реранжирование", 
     "text": "Ранжирование (ranking) — это процесс упорядочивания найденных документов по степени релевантности запросу. В векторном поиске начальное ранжирование выполняется на основе сходства эмбеддингов. Однако первичное ранжирование может быть неточным. Для улучшения качества применяется реранжирование (reranking). Реранжирование — это второй этап, на котором небольшая топ‑k результатов переоценивается более точной, но медленной моделью. Модели реранжирования часто основаны на кросс‑энкодерах (cross‑encoders), которые вычисляют совместное представление запроса и документа. Кросс‑энкодеры более точны, чем би‑энкодеры, но их нельзя использовать для поиска по всему корпусу из‑за вычислительной сложности. Типичный пайплайн: би‑энкодер (быстрый) ищет top‑k, затем кросс‑энкодер (медленный) реранжирует эти кандидаты. Реранжирование значительно повышает метрики retrieval (особенно MRR)."},
    {"id": 22, "title": "Гибридный поиск в RAG", 
     "text": "Гибридный поиск объединяет два подхода: dense поиск (на основе эмбеддингов) и sparse поиск (на основе TF‑IDF или BM25). Dense поиск хорошо находит семантически похожие документы, даже если в них нет точных ключевых слов. Sparse поиск превосходит dense в ситуациях, когда важны точные совпадения терминов (например, имена, коды, даты). Гибридный поиск комбинирует оценки релевантности от обоих методов с помощью взвешенной суммы или рангового слияния. Это позволяет получить результаты, которые учитывают и смысл, и точные совпадения. В RAG‑системах гибридный поиск часто даёт лучшие метрики, чем каждый из методов по отдельности."},
    {"id": 23, "title": "Производственные RAG‑системы", 
     "text": "Производственные (production) RAG‑системы предъявляют высокие требования к надёжности, скорости и масштабируемости. Они должны обрабатывать тысячи запросов в секунду с низкой задержкой (обычно менее 1 секунды). В таких системах векторные индексы часто строятся на основе HNSW или IVF с использованием FAISS или специализированных БД (Qdrant, Milvus, Pinecone). Чанкинг выполняется не по словам, а с учётом структуры документа (абзацы, заголовки). Эмбеддинги обычно получают с помощью специально обученных моделей, иногда с тонкой настройкой на доменной области. Для генерации ответов используются LLM, развёрнутые локально или через API. Production RAG включает мониторинг метрик и автоматические A/B тесты."}
]

print("ДО обновления базы знаний:")
query_new1 = "Что такое реранжирование?"
query_new2 = "В чём суть гибридного поиска?"
for q in [query_new1, query_new2]:
    res = search(q, index, chunks, top_k=3)
    titles = [r['chunk']['doc_title'] for r in res]
    print(f"Запрос: '{q}' -> найдены заголовки: {titles}")

updated_kb = knowledge_base + new_docs
updated_chunks = []
for doc in updated_kb:
    doc_chunks = chunk_text(doc['text'], CHUNK_SIZE, OVERLAP)
    for idx, chunk_text_str in enumerate(doc_chunks):
        updated_chunks.append({
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_index': idx,
            'text': chunk_text_str
        })

updated_texts = [c['text'] for c in updated_chunks]
updated_embeddings = model.encode(updated_texts, convert_to_numpy=True, show_progress_bar=True)
faiss.normalize_L2(updated_embeddings)
updated_index = faiss.IndexFlatIP(dim)
updated_index.add(updated_embeddings)

def search_updated(query, top_k=3):
    return search(query, updated_index, updated_chunks, top_k)

print("\nПОСЛЕ обновления базы знаний:")
for q in [query_new1, query_new2]:
    res = search_updated(q, top_k=3)
    titles = [r['chunk']['doc_title'] for r in res]
    print(f"Запрос: '{q}' -> найдены заголовки: {titles}")

comparison_data = []
for q, expected in [(query_new1, "Ранжирование в поиске и реранжирование"), 
                    (query_new2, "Гибридный поиск в RAG")]:
    before = search(q, index, chunks, top_k=3)
    after = search_updated(q, top_k=3)
    comparison_data.append({
        'query': q,
        'before_retrieved_sources': ', '.join([r['chunk']['doc_title'] for r in before]),
        'after_retrieved_sources': ', '.join([r['chunk']['doc_title'] for r in after]),
        'changed': expected in [r['chunk']['doc_title'] for r in after] and expected not in [r['chunk']['doc_title'] for r in before]
    })
df_compare = pd.DataFrame(comparison_data)
df_compare.to_csv('artifacts/retrieval_before_after_update.csv', index=False, encoding='utf-8-sig')
print("\nСравнение сохранено в artifacts/retrieval_before_after_update.csv")

ДО обновления базы знаний:
Запрос: 'Что такое реранжирование?' -> найдены заголовки: ['Ранжирование в поиске и реранжирование', 'Чанкинг и overlap при разбиении текста', 'Retrieval-Augmented Generation (RAG)']
Запрос: 'В чём суть гибридного поиска?' -> найдены заголовки: ['Гибридный поиск в RAG', 'Гибридный поиск в RAG', 'Гибридный поиск в RAG']


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


ПОСЛЕ обновления базы знаний:
Запрос: 'Что такое реранжирование?' -> найдены заголовки: ['Ранжирование в поиске и реранжирование', 'Чанкинг и overlap при разбиении текста', 'Retrieval-Augmented Generation (RAG)']
Запрос: 'В чём суть гибридного поиска?' -> найдены заголовки: ['Гибридный поиск в RAG', 'Гибридный поиск в RAG', 'Гибридный поиск в RAG']

Сравнение сохранено в artifacts/retrieval_before_after_update.csv


In [14]:
all_chunk_texts = [c['text'] for c in updated_chunks]
tfidf_vectorizer = TfidfVectorizer().fit(all_chunk_texts)

def extractive_answer(query, top_k=3):
    retrieved = search_updated(query, top_k=top_k)
    sources = list(set([r['chunk']['doc_title'] for r in retrieved]))
    context = " ".join([r['chunk']['text'] for r in retrieved])
    sentences = [s.strip() for s in context.split('.') if len(s.strip()) > 10]
    if not sentences:
        return "Не удалось сформировать ответ.", sources
    query_vec = tfidf_vectorizer.transform([query])
    sent_vecs = tfidf_vectorizer.transform(sentences)
    similarities = cosine_similarity(query_vec, sent_vecs).flatten()
    best_idx = np.argmax(similarities)
    answer = sentences[best_idx] + "."
    return answer, sources

demo_queries = [
    "Как измерить семантическую близость текстов?",
    "Что такое RAG и зачем он нужен?",
    "Какие метрики использовать для оценки поиска?"
]
print("Примеры работы mini-RAG:\n")
for q in demo_queries:
    ans, src = extractive_answer(q, top_k=2)
    print(f"Вопрос: {q}")
    print(f"Ответ: {ans}")
    print(f"Источники: {', '.join(src)}\n")

rag_examples = []
test_rag_queries = demo_queries + [
    "Как работает FAISS?",
    "Что такое overlap при чанкинге?",
    "Как обновить базу знаний?"
]
for q in test_rag_queries:
    ans, src = extractive_answer(q, top_k=2)
    rag_examples.append({
        'question': q,
        'answer': ans,
        'retrieved_sources': ', '.join(src)
    })
df_rag = pd.DataFrame(rag_examples)
df_rag.to_csv('artifacts/rag_examples.csv', index=False, encoding='utf-8-sig')

Примеры работы mini-RAG:

Вопрос: Как измерить семантическую близость текстов?
Ответ: Эмбеддинги текстов применяются в задачах семантического поиска, кластеризации, классификации и перефразирования.
Источники: Эмбеддинги текстов, Чанкинг и overlap при разбиении текста

Вопрос: Что такое RAG и зачем он нужен?
Ответ: Качество RAG сильно зависит от качества.
Источники: Эмбеддинги текстов, Retrieval-Augmented Generation (RAG)

Вопрос: Какие метрики использовать для оценки поиска?
Ответ: Для реализации гибридного поиска можно использовать FAISS для dense части и отдельный индекс (например, на основе whoosh или панды) для sparse.
Источники: Гибридный поиск в RAG



In [10]:
error_queries = [
    "Какая сегодня погода?",               
    "Что такое индекс в базе данных SQL?",  
    "Как нормализовать эмбеддинги?"         
]

print("Анализ ошибок:\n")
for q in error_queries:
    ans, src = extractive_answer(q, top_k=2)
    print(f"Запрос: {q}")
    print(f"Ответ: {ans}")
    print(f"Источники: {src}\n")
    if "погода" in q:
        print("  -> Проблема: запрос вне предметной области. Retrieval возвращает случайные чанки, ответ бессмыслен.")
    elif "SQL" in q:
        print("  -> Проблема: в базе знаний нет информации об индексах SQL. Retrieval находит ближайший по смыслу чанк про FAISS, но ответ неверен.")
    elif "нормализовать" in q:
        print("  -> Проблема: ответ есть в документе 'Эмбеддинги текстов', но экстрактивный генератор выбрал не то предложение из-за шума в контексте.")
    print("---")

print("Общие выводы:")
print("- Внепредметные запросы приводят к случайным ответам (нужна фильтрация).")
print("- Если база знаний не покрывает тему, RAG не может дать верный ответ.")
print("- Экстрактивный генератор сильно зависит от формулировки и выбора предложения.")
print("- Улучшение retrieval (реранжирование) и замена генератора на LLM могли бы исправить многие ошибки.")

Анализ ошибок:

Запрос: Какая сегодня погода?
Ответ: хорошее задание для самостоятельной работы.
Источники: ['TF‑IDF как альтернатива эмбеддингам', 'Ранжирование в поиске и реранжирование']

  -> Проблема: запрос вне предметной области. Retrieval возвращает случайные чанки, ответ бессмыслен.
---
Запрос: Что такое индекс в базе данных SQL?
Ответ: Эти методы значительно ускоряют поиск на больших наборах данных за.
Источники: ['Индексирование в FAISS: типы индексов', 'FAISS и поиск ближайших соседей']

  -> Проблема: в базе знаний нет информации об индексах SQL. Retrieval находит ближайший по смыслу чанк про FAISS, но ответ неверен.
---
Запрос: Как нормализовать эмбеддинги?
Ответ: позволяет легко нормализовать эмбеддинги с помощью метода encode(.
Источники: ['Sentence‑transformers модели для русского языка', 'Ранжирование в поиске и реранжирование']

  -> Проблема: ответ есть в документе 'Эмбеддинги текстов', но экстрактивный генератор выбрал не то предложение из-за шума в контексте.
---
